In [1]:
pip install "datasets==3.6.0" "numpy<2"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from datasets import load_dataset

doc_data = load_dataset("parquet", data_files="hf://datasets/IBM/multidoc2dial@~parquet/document_domain/train/*.parquet")
print(doc_data)
sample_doc = doc_data["train"][0]
print("Top-level keys:", sample_doc.keys())
print("doc_id:", sample_doc["doc_id"])
print("title:", sample_doc["title"])
print("domain:", sample_doc["domain"])
print("Number of spans:", len(sample_doc["spans"]))
print("First span:", sample_doc["spans"][0])

DatasetDict({
    train: Dataset({
        features: ['domain', 'doc_id', 'title', 'doc_text', 'spans', 'doc_html_ts', 'doc_html_raw'],
        num_rows: 488
    })
})
Top-level keys: dict_keys(['domain', 'doc_id', 'title', 'doc_text', 'spans', 'doc_html_ts', 'doc_html_raw'])
doc_id: Benefits Planner: Survivors | Planning For Your Survivors | Social Security Administration#1_0
title: Benefits Planner: Survivors | Planning For Your Survivors | Social Security Administration#1
domain: ssa
Number of spans: 99
First span: {'id_sp': '1', 'tag': 'h2', 'start_sp': 0, 'end_sp': 61, 'text_sp': '\n\nBenefits Planner: Survivors | Planning For Your Survivors \n', 'title': 'Benefits Planner: Survivors | Planning For Your Survivors', 'parent_titles': {'id_sp': [], 'text': [], 'level': []}, 'id_sec': 't_0', 'start_sec': 0, 'text_sec': '\n\nBenefits Planner: Survivors | Planning For Your Survivors \n', 'end_sec': 61}


In [3]:
import numpy as np

span_lengths = [len(sp["text_sp"]) for doc in doc_data["train"] for sp in doc["spans"]]
print("Total spans:", len(span_lengths))
print("Mean/median/min/max span length (chars):",
      np.mean(span_lengths), np.median(span_lengths), min(span_lengths), max(span_lengths))

sections = set(sp["id_sec"] for sp in sample_doc["spans"])
print("Unique section ids in one sample doc:", sections)

Total spans: 35659
Mean/median/min/max span length (chars): 64.90964412911187 54.0 2 439
Unique section ids in one sample doc: {'6', '7', '31', '13', 't_12', '14', '16', '17', '23', '20', 't_8', '9', '24', 't_5', 't_17', '22', '4', '11', 't_23', '21', 't_0', '26', '30', '8', '27', 't_20', '3', 't_15', '25', '28', '18', '2', '5', '10', '19', '12', '29', '15', '1'}


In [4]:
from collections import defaultdict
import numpy as np

def group_spans_by_section(doc):
    sections = defaultdict(list)
    for sp in doc["spans"]:
        sections[sp["id_sec"]].append(sp)
    grouped = []
    for sec_id, spans in sections.items():
        spans_sorted = sorted(spans, key=lambda s: s["start_sp"])
        text = "".join(s["text_sp"] for s in spans_sorted)
        grouped.append({
            "doc_id": doc["doc_id"],
            "id_sec": sec_id,
            "span_ids": [s["id_sp"] for s in spans_sorted],
            "text": text,
        })
    return grouped

sample_sections = group_spans_by_section(sample_doc)
print("Sections in sample doc:", len(sample_sections))
for s in sample_sections[:5]:
    print(s["id_sec"], "| len:", len(s["text"]), "|", repr(s["text"][:120]))

Sections in sample doc: 39
t_0 | len: 61 | '\n\nBenefits Planner: Survivors | Planning For Your Survivors \n'
1 | len: 213 | "As you plan for the future , you'll want to think about what your family would need if you should die now. Social Securi"
2 | len: 219 | 'You can earn up to four credits each year. In 2019 , for example , you earn one credit for each $1,360 of wages or self '
3 | len: 316 | 'The number of credits needed to provide benefits for your survivors depends on your age when you die. No one needs more '
4 | len: 271 | "Benefits can be paid to your children and your spouse who is caring for the children even if you don't have the required"


In [5]:
all_section_lengths = []
for doc in doc_data["train"]:
    secs = group_spans_by_section(doc)
    all_section_lengths.extend(len(s["text"]) for s in secs)

print("Total sections across KB:", len(all_section_lengths))
print("Mean/median/min/max section length (chars):",
      np.mean(all_section_lengths), np.median(all_section_lengths),
      min(all_section_lengths), max(all_section_lengths))

Total sections across KB: 13421
Mean/median/min/max section length (chars): 172.46203710602788 112.0 4 3341


In [6]:
def build_chunks(doc, target_min=200, target_max=600):
    sections = defaultdict(list)
    order = []
    for sp in doc["spans"]:
        if sp["id_sec"] not in sections:
            order.append(sp["id_sec"])
        sections[sp["id_sec"]].append(sp)

    section_texts = []
    for sec_id in order:
        spans_sorted = sorted(sections[sec_id], key=lambda s: s["start_sp"])
        text = "".join(s["text_sp"] for s in spans_sorted).strip()
        is_heading = sec_id.startswith("t_")
        section_texts.append({
            "id_sec": sec_id,
            "span_ids": [s["id_sp"] for s in spans_sorted],
            "text": text,
            "is_heading": is_heading,
        })

    chunks = []
    current_heading = ""
    current_heading_span_ids = []
    buffer_text = []
    buffer_spans = []
    buffer_len = 0

    def flush(force=False):
        nonlocal buffer_text, buffer_spans, buffer_len, current_heading_span_ids
        has_content = bool(buffer_text)
        has_leftover_heading = force and bool(current_heading_span_ids) and not has_content
        if has_content or has_leftover_heading:
            full_text = (current_heading + "\n" if current_heading else "") + "\n".join(buffer_text)
            chunks.append({
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "domain": doc["domain"],
                "heading": current_heading,
                "span_ids": current_heading_span_ids + list(buffer_spans),
                "text": full_text.strip(),
            })
            current_heading_span_ids = []
        buffer_text, buffer_spans, buffer_len = [], [], 0

    for sec in section_texts:
        if sec["is_heading"]:
            if buffer_text and buffer_len >= target_min:
                # enough real content already — flush it, start a fresh chunk under this heading
                flush()
                current_heading = sec["text"]
                current_heading_span_ids = list(sec["span_ids"])
            elif buffer_text:
                # content exists but below target_min — treat this heading as an inline
                # sub-heading within the SAME growing chunk, don't cut here
                buffer_text.append(sec["text"])
                buffer_spans.extend(sec["span_ids"])
                buffer_len += len(sec["text"])
            else:
                # no content since the last flush — consecutive heading, chain it
                current_heading = sec["text"]
                current_heading_span_ids = current_heading_span_ids + list(sec["span_ids"])
            continue

        if buffer_len + len(sec["text"]) > target_max and buffer_len >= target_min:
            flush()
        buffer_text.append(sec["text"])
        buffer_spans.extend(sec["span_ids"])
        buffer_len += len(sec["text"])

    flush(force=True)
    return chunks

In [7]:
all_chunks = []
for doc in doc_data["train"]:
    all_chunks.extend(build_chunks(doc))

lengths = [len(c["text"]) for c in all_chunks]
print("Total chunks:", len(all_chunks))
print("Mean/median/min/max chunk length (chars):",
      np.mean(lengths), np.median(lengths), min(lengths), max(lengths))

Total chunks: 4799
Mean/median/min/max chunk length (chars): 496.3431964992707 478.0 16 3402


In [8]:
import json

all_chunks = []
for doc in doc_data["train"]:
    all_chunks.extend(build_chunks(doc))

for i, c in enumerate(all_chunks):
    c["chunk_id"] = f"{c['doc_id']}::chunk_{i}"

with open("../data/processed/kb_chunks.jsonl", "w", encoding="utf-8") as f:
    for c in all_chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

print("Saved", len(all_chunks), "chunks")

Saved 4799 chunks


In [9]:
dial_data = load_dataset("parquet", data_files="hf://datasets/IBM/multidoc2dial@~parquet/dialogue_domain/train/*.parquet")
print(dial_data)
sample_dial = dial_data["train"][0]
print(sample_dial.keys())
print("domain:", sample_dial["domain"], "| dial_id:", sample_dial["dial_id"])

roles = set()
for turn in sample_dial["turns"]:
    roles.add(turn["role"])
print("Roles seen:", roles)

for turn in sample_dial["turns"][:4]:
    print(turn["role"], "|", turn["utterance"][:100])
    print("  references:", turn["references"])

DatasetDict({
    train: Dataset({
        features: ['dial_id', 'domain', 'turns'],
        num_rows: 3474
    })
})
dict_keys(['dial_id', 'domain', 'turns'])
domain: dmv | dial_id: 8df07b7a98990db27c395cb1f68a962e
Roles seen: {'agent', 'user'}
user | Hello, I forgot o update my address, can you help me with that?
  references: [{'id_sp': '4', 'label': 'precondition', 'doc_id': 'Top 5 DMV Mistakes and How to Avoid Them#3_0'}]
agent | hi, you have to report any change of address to DMV within 10 days after moving. You should do this 
  references: [{'id_sp': '6', 'label': 'solution', 'doc_id': 'Top 5 DMV Mistakes and How to Avoid Them#3_0'}, {'id_sp': '7', 'label': 'solution', 'doc_id': 'Top 5 DMV Mistakes and How to Avoid Them#3_0'}]
user | Can I do my DMV transactions online?
  references: [{'id_sp': '56', 'label': 'solution', 'doc_id': 'Top 5 DMV Mistakes and How to Avoid Them#3_0'}]
agent | Yes, you can sign up for MyDMV for all the online transactions needed.
  references: [{'id_s

In [10]:
def build_span_to_chunk_lookup(chunks):
    lookup = {}
    for c in chunks:
        for sp_id in c["span_ids"]:
            lookup[(c["doc_id"], sp_id)] = c["chunk_id"]
    return lookup

span_to_chunk = build_span_to_chunk_lookup(all_chunks)
print("Span->chunk lookup size:", len(span_to_chunk))

qa_pairs = []
resolved, unresolved = 0, 0

for dial in dial_data["train"]:
    turns = dial["turns"]
    for i, turn in enumerate(turns):
        if turn["role"] != "agent" or not turn["references"]:
            continue
        prev_user_turns = [t for t in turns[:i] if t["role"] == "user"]
        if not prev_user_turns:
            continue
        query = prev_user_turns[-1]["utterance"]

        gold_chunks = set()
        for ref in turn["references"]:
            key = (ref["doc_id"], ref["id_sp"])
            if key in span_to_chunk:
                gold_chunks.add(span_to_chunk[key])
                resolved += 1
            else:
                unresolved += 1

        if gold_chunks:
            qa_pairs.append({
                "dial_id": dial["dial_id"],
                "domain": dial["domain"],
                "query": query,
                "answer_utterance": turn["utterance"],
                "gold_chunk_ids": sorted(gold_chunks),
            })

print(f"QA pairs built: {len(qa_pairs)}")
print(f"Resolved refs: {resolved}, Unresolved: {unresolved}")
print(f"Resolution rate: {resolved / (resolved + unresolved):.2%}")

Span->chunk lookup size: 35659
QA pairs built: 24603
Resolved refs: 39304, Unresolved: 0
Resolution rate: 100.00%


Splitting

In [11]:
import random

random.seed(42)

dial_ids = list(set(p["dial_id"] for p in qa_pairs))
random.shuffle(dial_ids)

n = len(dial_ids)
train_ids = set(dial_ids[:int(0.8 * n)])
dev_ids = set(dial_ids[int(0.8 * n):int(0.9 * n)])
test_ids = set(dial_ids[int(0.9 * n):])

train_pairs = [p for p in qa_pairs if p["dial_id"] in train_ids]
dev_pairs = [p for p in qa_pairs if p["dial_id"] in dev_ids]
test_pairs = [p for p in qa_pairs if p["dial_id"] in test_ids]

print(f"Dialogues: train={len(train_ids)}, dev={len(dev_ids)}, test={len(test_ids)}")
print(f"QA pairs: train={len(train_pairs)}, dev={len(dev_pairs)}, test={len(test_pairs)}")

for name, pairs in [("train", train_pairs), ("dev", dev_pairs), ("test", test_pairs)]:
    with open(f"../data/processed/qa_pairs_{name}.jsonl", "w", encoding="utf-8") as f:
        for p in pairs:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")

print("Saved train/dev/test splits.")

Dialogues: train=2779, dev=347, test=348
QA pairs: train=19699, dev=2434, test=2470
Saved train/dev/test splits.


In [12]:
from sentence_transformers import SentenceTransformer
import numpy as np

baseline_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [c["text"] for c in all_chunks]
chunk_ids = [c["chunk_id"] for c in all_chunks]
chunk_embeddings = baseline_model.encode(chunk_texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)

print("Chunk embeddings shape:", chunk_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/75 [00:00<?, ?it/s]

Chunk embeddings shape: (4799, 384)


In [13]:
def evaluate_retrieval(model, chunk_embeddings, chunk_ids, eval_pairs, k_values=(1, 5, 10)):
    queries = [p["query"] for p in eval_pairs]
    query_embeddings = model.encode(queries, batch_size=64, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)

    sims = query_embeddings @ chunk_embeddings.T  # cosine sim since normalized
    ranked_idx = np.argsort(-sims, axis=1)

    recalls = {k: 0 for k in k_values}
    mrr_total = 0.0

    for i, pair in enumerate(eval_pairs):
        gold_set = set(pair["gold_chunk_ids"])
        ranked_chunk_ids = [chunk_ids[idx] for idx in ranked_idx[i]]

        # MRR: rank of the first gold chunk found
        rank = next((r + 1 for r, cid in enumerate(ranked_chunk_ids) if cid in gold_set), None)
        if rank is not None:
            mrr_total += 1.0 / rank

        for k in k_values:
            if any(cid in gold_set for cid in ranked_chunk_ids[:k]):
                recalls[k] += 1

    n = len(eval_pairs)
    results = {f"Recall@{k}": recalls[k] / n for k in k_values}
    results["MRR"] = mrr_total / n
    return results

baseline_results = evaluate_retrieval(baseline_model, chunk_embeddings, chunk_ids, dev_pairs)
print("Baseline (untuned MiniLM) on dev set:")
for k, v in baseline_results.items():
    print(f"  {k}: {v:.4f}")

Batches:   0%|          | 0/39 [00:00<?, ?it/s]

Baseline (untuned MiniLM) on dev set:
  Recall@1: 0.1795
  Recall@5: 0.3550
  Recall@10: 0.4306
  MRR: 0.2652


In [14]:
chunks_by_domain = defaultdict(list)
for c in all_chunks:
    chunks_by_domain[c["domain"]].append(c["chunk_id"])

def sample_hard_negative(pair, chunk_id_set_by_domain, n=1):
    domain_pool = chunk_id_set_by_domain[pair["domain"]]
    gold_set = set(pair["gold_chunk_ids"])
    negatives = []
    attempts = 0
    while len(negatives) < n and attempts < 20:
        candidate = random.choice(domain_pool)
        if candidate not in gold_set:
            negatives.append(candidate)
        attempts += 1
    return negatives

In [15]:
from datasets import Dataset

chunk_text_by_id = {c["chunk_id"]: c["text"] for c in all_chunks}

def build_contrastive_dataset(pairs, chunks_by_domain, n_negatives=1):
    anchors, positives, negatives = [], [], []
    for pair in pairs:
        neg_ids = sample_hard_negative(pair, chunks_by_domain, n=n_negatives)
        if not neg_ids:
            continue
        for gold_id in pair["gold_chunk_ids"][:1]:  # one positive per pair, keeps it simple
            for neg_id in neg_ids:
                anchors.append(pair["query"])
                positives.append(chunk_text_by_id[gold_id])
                negatives.append(chunk_text_by_id[neg_id])

    return Dataset.from_dict({"anchor": anchors, "positive": positives, "negative": negatives})

train_dataset = build_contrastive_dataset(train_pairs, chunks_by_domain, n_negatives=1)
print("Training triplets:", len(train_dataset))
print(train_dataset[0])

Training triplets: 19699
{'anchor': 'Hello, I forgot o update my address, can you help me with that?', 'positive': '1. Forgetting to Update Address\nBy statute , you must report a change of address to DMV within ten days of moving. That is the case for the address associated with your license, as well as all the addresses associated with each registered vehicle, which may differ.', 'negative': 'How do I know what my current voter registration status is?\nPlease note: The DMV does not approve or deny voter registration applications. We only send the application to the County or City Board of Elections for their review.\nIf you do not want to change the information on your DMV records but still want to register to vote or update your voter registration information , complete the New York State Voter Registration Form PDF on the New York State Board of Elections website [2] and mail it to your County Board of Elections.\nUse the MV Electronic Voter Registration application [6 ]'}


In [16]:
lengths = [len(c["text"]) for c in all_chunks]
short_chunks = [c for c in all_chunks if len(c["text"]) < 60]

print(f"Chunks under 60 chars: {len(short_chunks)} / {len(all_chunks)} ({len(short_chunks)/len(all_chunks):.2%})")
print()
for c in short_chunks[:15]:
    print(repr(c["text"]))
    print("---")

Chunks under 60 chars: 19 / 4799 (0.40%)

'Create a my Social Security'
---
'Create a my Social Security'
---
'How long does it take VA to make a decision?'
---
'By fax\nFax your request to 844 - 678 - 8979.'
---
'Topic:\nMyDMV [9]'
---
'Topic:\nMyDMV [4]'
---
'yes or no survey:'
---
'yes or no survey:'
---
'yes or no survey:'
---
'Topic:\nYounger Driver [4 ]'
---
'Topic:\nMyDMV [2]'
---
'Disable this transaction?:'
---
'yes or no survey:'
---
'yes or no survey:'
---
'Topic:\nContact Us [6 ]'
---


In [17]:
from collections import Counter

text_counts = Counter(c["text"] for c in all_chunks)
duplicated = [(text, count) for text, count in text_counts.items() if count > 1]
duplicated.sort(key=lambda x: -x[1])

print(f"Distinct chunk texts that repeat across the KB: {len(duplicated)}")
for text, count in duplicated[:15]:
    print(count, "times:", repr(text))

Distinct chunk texts that repeat across the KB: 465
20 times: 'What original documents do I need?\nIf you do not have one of these specific documents or you cannot get a replacement for one of them within 10 days , we will ask to see other documents. Any documents submitted, including the following, must be current not expired and show your name, identifying information date of birth or age and preferably a recent photograph : Employee identification card ; School identification card ; Health insurance card not a Medicare card ; or U.S. military identification card.\nNote'
15 times: "What original documents do I need?\nYou : We also must see proof of your identity. An acceptable document must be current not expired and show your name, identifying information date of birth or age and preferably a recent photograph. For example , as proof of identity Social Security must see your : U.S. driver's license ; State - issued non - driver identification card ; or U.S. passport."
9 times: "What

In [18]:
# rebuild with the fixed chunker first
all_chunks = []
for doc in doc_data["train"]:
    all_chunks.extend(build_chunks(doc))
for i, c in enumerate(all_chunks):
    c["chunk_id"] = f"{c['doc_id']}::chunk_{i}"

print("Chunks after flush() fix:", len(all_chunks))
short_chunks = [c for c in all_chunks if len(c["text"]) < 60]
print(f"Still under 60 chars: {len(short_chunks)} / {len(all_chunks)} ({len(short_chunks)/len(all_chunks):.2%})")

# now check duplication spread specifically for short texts
text_to_docs = defaultdict(set)
for c in all_chunks:
    text_to_docs[c["text"]].add(c["doc_id"])

MIN_CONTENT_LEN = 80
MIN_DOC_SPREAD = 3

boilerplate_candidates = {
    t for t, docs in text_to_docs.items()
    if len(t) < MIN_CONTENT_LEN and len(docs) >= MIN_DOC_SPREAD
}
print(f"\nCandidate boilerplate texts (short + spread across >= {MIN_DOC_SPREAD} docs): {len(boilerplate_candidates)}")
for t in sorted(boilerplate_candidates, key=lambda x: -len(text_to_docs[x]))[:20]:
    print(f"  ({len(text_to_docs[t])} docs) {t!r}")

Chunks after flush() fix: 4799
Still under 60 chars: 19 / 4799 (0.40%)

Candidate boilerplate texts (short + spread across >= 3 docs): 1
  (5 docs) 'yes or no survey:'


In [19]:
remaining_short = [c for c in all_chunks if len(c["text"]) < 60]
for c in remaining_short:
    print(repr(c["text"]), "| doc:", c["doc_id"][:50])
    print("---")

'Create a my Social Security' | doc: Supplemental Security Income (SSI) Benefits | Soci
---
'Create a my Social Security' | doc: Supplemental Security Income (SSI) Benefits | Soci
---
'How long does it take VA to make a decision?' | doc: How To Apply For The GI Bill | Veterans Affairs#1_
---
'By fax\nFax your request to 844 - 678 - 8979.' | doc: Board hearings with a Veterans Law Judge | Veteran
---
'Topic:\nMyDMV [9]' | doc: Creating a MyDMV account#1_0
---
'Topic:\nMyDMV [4]' | doc: Signing in to MyDMV#1_0
---
'yes or no survey:' | doc: Get a military skills test waiver#1_0
---
'yes or no survey:' | doc: How to check a title or lien status#1_0
---
'yes or no survey:' | doc: Exchange your out-of-state driver license#1_0
---
'Topic:\nYounger Driver [4 ]' | doc: The Graduated License Law and Restrictions for Dri
---
'Topic:\nMyDMV [2]' | doc: MyDMV Account Terms of Service#1_0
---
'Disable this transaction?:' | doc: Voter Registration Application Frequently Asked Qu
---
'yes or no surve

In [20]:
import re

junk_exact = {
    "Create a my Social Security",
    "yes or no survey:",
    "Disable this transaction?:",
    "Didn't find what you're looking for?",
    "Apply for a PLUS Loan\nLOG IN TO START\nStart Demo",
    "Estimate Your Aid\nGet Aid Estimate",
    "How long does it take VA to make a decision?",
}
topic_tag_pattern = re.compile(r"^Topic:\n.*\[\d+\s*\]$")

def is_junk(text):
    text = text.strip()
    if text in junk_exact:
        return True
    if topic_tag_pattern.match(text):
        return True
    return False

junk_chunks = [c for c in all_chunks if is_junk(c["text"])]
print(f"Total junk chunks flagged: {len(junk_chunks)} / {len(all_chunks)}")
for c in junk_chunks:
    print(repr(c["text"][:70]))

Total junk chunks flagged: 18 / 4799
'Create a my Social Security'
'Create a my Social Security'
'How long does it take VA to make a decision?'
'Topic:\nMyDMV [9]'
'Topic:\nMyDMV [4]'
'yes or no survey:'
'yes or no survey:'
'yes or no survey:'
'Topic:\nYounger Driver [4 ]'
'Topic:\nMyDMV [2]'
'Disable this transaction?:'
'yes or no survey:'
'yes or no survey:'
'Topic:\nContact Us [6 ]'
"Didn't find what you're looking for?"
'Apply for a PLUS Loan\nLOG IN TO START\nStart Demo'
'Estimate Your Aid\nGet Aid Estimate'
'Apply for a PLUS Loan\nLOG IN TO START\nStart Demo'


In [21]:
clean_chunks = [c for c in all_chunks if not is_junk(c["text"])]
print(f"Chunks kept: {len(clean_chunks)} (removed {len(all_chunks) - len(clean_chunks)})")

for i, c in enumerate(clean_chunks):
    c["chunk_id"] = f"{c['doc_id']}::chunk_{i}"
all_chunks = clean_chunks

with open("../data/processed/kb_chunks.jsonl", "w", encoding="utf-8") as f:
    for c in all_chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")
print("Saved cleaned KB:", len(all_chunks), "chunks")

# rebuild lookup and re-run the dialogue join to confirm nothing broke
span_to_chunk = build_span_to_chunk_lookup(all_chunks)

qa_pairs = []
resolved, unresolved = 0, 0
for dial in dial_data["train"]:
    turns = dial["turns"]
    for i, turn in enumerate(turns):
        if turn["role"] != "agent" or not turn["references"]:
            continue
        prev_user_turns = [t for t in turns[:i] if t["role"] == "user"]
        if not prev_user_turns:
            continue
        query = prev_user_turns[-1]["utterance"]
        gold_chunks = set()
        for ref in turn["references"]:
            key = (ref["doc_id"], ref["id_sp"])
            if key in span_to_chunk:
                gold_chunks.add(span_to_chunk[key])
                resolved += 1
            else:
                unresolved += 1
        if gold_chunks:
            qa_pairs.append({
                "dial_id": dial["dial_id"], "domain": dial["domain"],
                "query": query, "answer_utterance": turn["utterance"],
                "gold_chunk_ids": sorted(gold_chunks),
            })

print(f"QA pairs built: {len(qa_pairs)}")
print(f"Resolved refs: {resolved}, Unresolved: {unresolved}")
print(f"Resolution rate: {resolved / (resolved + unresolved):.2%}")

Chunks kept: 4781 (removed 18)
Saved cleaned KB: 4781 chunks
QA pairs built: 24569
Resolved refs: 39270, Unresolved: 34
Resolution rate: 99.91%


In [22]:
junk_span_keys = set()
for c in junk_chunks:
    for sp_id in c["span_ids"]:
        junk_span_keys.add((c["doc_id"], sp_id))

unresolved_matches_junk = 0
unresolved_other = []

for dial in dial_data["train"]:
    for turn in dial["turns"]:
        if turn["role"] != "agent":
            continue
        for ref in turn["references"]:
            key = (ref["doc_id"], ref["id_sp"])
            if key not in span_to_chunk:
                if key in junk_span_keys:
                    unresolved_matches_junk += 1
                else:
                    unresolved_other.append(key)

print(f"Unresolved refs explained by junk removal: {unresolved_matches_junk}")
print(f"Unresolved refs NOT explained by junk removal: {len(unresolved_other)}")
print(unresolved_other[:10])

Unresolved refs explained by junk removal: 34
Unresolved refs NOT explained by junk removal: 0
[]


In [29]:
random.seed(42)

dial_ids = sorted(set(p["dial_id"] for p in qa_pairs))
random.shuffle(dial_ids)

n = len(dial_ids)
train_ids = set(dial_ids[:int(0.8 * n)])
dev_ids = set(dial_ids[int(0.8 * n):int(0.9 * n)])
test_ids = set(dial_ids[int(0.9 * n):])

train_pairs = [p for p in qa_pairs if p["dial_id"] in train_ids]
dev_pairs = [p for p in qa_pairs if p["dial_id"] in dev_ids]
test_pairs = [p for p in qa_pairs if p["dial_id"] in test_ids]

print(f"Dialogues: train={len(train_ids)}, dev={len(dev_ids)}, test={len(test_ids)}")
print(f"QA pairs: train={len(train_pairs)}, dev={len(dev_pairs)}, test={len(test_pairs)}")

for name, pairs in [("train", train_pairs), ("dev", dev_pairs), ("test", test_pairs)]:
    with open(f"../data/processed/qa_pairs_{name}.jsonl", "w", encoding="utf-8") as f:
        for p in pairs:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")

print("Saved final train/dev/test splits.")

Dialogues: train=2779, dev=347, test=348
QA pairs: train=19640, dev=2461, test=2468
Saved final train/dev/test splits.


In [30]:
chunk_texts = [c["text"] for c in all_chunks]
chunk_ids = [c["chunk_id"] for c in all_chunks]
chunk_embeddings = baseline_model.encode(chunk_texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)

baseline_results = evaluate_retrieval(baseline_model, chunk_embeddings, chunk_ids, dev_pairs)
print("Baseline (untuned MiniLM) on cleaned KB / dev set:")
for k, v in baseline_results.items():
    print(f"  {k}: {v:.4f}")

Batches:   0%|          | 0/75 [00:00<?, ?it/s]

Batches:   0%|          | 0/39 [00:00<?, ?it/s]

Baseline (untuned MiniLM) on cleaned KB / dev set:
  Recall@1: 0.1853
  Recall@5: 0.3864
  Recall@10: 0.4624
  MRR: 0.2812


In [31]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0))

# sentence-transformers auto-picks a device on load — confirm what it actually chose
finetune_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model is on device:", finetune_model.device)

CUDA available: True
Device name: NVIDIA GeForce RTX 5050 Laptop GPU


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model is on device: cuda:0


In [32]:
# rebuild from the CURRENT, cleaned all_chunks / train_pairs
chunk_text_by_id = {c["chunk_id"]: c["text"] for c in all_chunks}
chunks_by_domain = defaultdict(list)
for c in all_chunks:
    chunks_by_domain[c["domain"]].append(c["chunk_id"])

train_dataset = build_contrastive_dataset(train_pairs, chunks_by_domain, n_negatives=1)
print("Training triplets (rebuilt, post-cleanup):", len(train_dataset))

Training triplets (rebuilt, post-cleanup): 19640


In [33]:
finetuned_model = SentenceTransformer("../data/indexes/retriever_finetuned_v1")
print("Loaded fine-tuned model from disk. Device:", finetuned_model.device)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded fine-tuned model from disk. Device: cuda:0


In [28]:
chunk_texts = [c["text"] for c in all_chunks]
chunk_ids = [c["chunk_id"] for c in all_chunks]

finetuned_embeddings = finetuned_model.encode(
    chunk_texts, batch_size=64, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True
)

finetuned_results = evaluate_retrieval(finetuned_model, finetuned_embeddings, chunk_ids, dev_pairs)

print("Fine-tuned retriever on cleaned KB / dev set:")
for k, v in finetuned_results.items():
    print(f"  {k}: {v:.4f}")

print("\nBaseline (untuned) for comparison:")
for k, v in baseline_results.items():
    print(f"  {k}: {v:.4f}")

Batches:   0%|          | 0/75 [00:00<?, ?it/s]

Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Fine-tuned retriever on cleaned KB / dev set:
  Recall@1: 0.2719
  Recall@5: 0.5356
  Recall@10: 0.6290
  MRR: 0.3928

Baseline (untuned) for comparison:
  Recall@1: 0.1822
  Recall@5: 0.3558
  Recall@10: 0.4319
  MRR: 0.2672
